# InferStack: first A100 qualification

Upload to Colab, choose Runtime > Change runtime type > A100 GPU, then run cells in order. GPU availability is not guaranteed. Use a read-only GitHub token when prompted. This notebook pins the accepted H0 source and the existing Foundry model. No vLLM, SGLang or paid optimizer API is used. Native dependencies and runtime torch/CUDA versions are recorded.

Prepared from source and syntax-checked locally; not GPU-executed by the author. If any setup cell fails, inspect its log and stop rather than bypassing assertions. Sandbox limitations can produce not_qualified even when numerical/model tests run; preserve that outcome. Logs and a downloadable ZIP are the deliverable.


## 1. Verify GPU, clone privately, and pin source


In [ ]:
import base64, datetime, getpass, json, os, signal, subprocess, sys, time
from pathlib import Path
PLATFORM = 'colab'
COMMIT = 'd33b47032de1427a45d662fae05fffb09d5dcfea'
BASE = Path('/content' if PLATFORM == 'colab' else '/kaggle/working')
REPO = BASE / 'inferstack'
assert (3, 11) <= sys.version_info[:2] < (3, 14), 'This profile needs Python 3.11–3.13.'
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True)
print(gpu)
if PLATFORM == 'colab':
    assert 'A100' in gpu, 'Choose an A100 runtime for this notebook. Use the separate T4 notebook for Kaggle.'
else:
    assert len([line for line in gpu.splitlines() if 'T4' in line]) >= 2, 'Choose Kaggle GPU T4 x2.'

def clone_pinned():
    if not (REPO / '.git').is_dir():
        if PLATFORM == 'kaggle':
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret('GITHUB_TOKEN').strip()
        else:
            token = getpass.getpass('Read-only GitHub token for this private repo: ').strip()
        env = os.environ.copy()
        env.update(GIT_TERMINAL_PROMPT='0', GIT_CONFIG_COUNT='1',
                   GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
                   GIT_CONFIG_VALUE_0='AUTHORIZATION: basic '+base64.b64encode(('x-access-token:'+token).encode()).decode())
        result = subprocess.run(['git','clone','-q','https://github.com/sanjoxtech/inferstack.git',str(REPO)],
                                env=env,capture_output=True,text=True)
        del token, env
        if result.returncode:
            raise RuntimeError('Clone failed. Check read-only repository access and Internet connectivity. Credentials were not printed or stored.')
    subprocess.run(['git','checkout','--detach',COMMIT],cwd=REPO,check=True,capture_output=True,text=True)
    actual=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
    assert actual == COMMIT
    print('Pinned source:', actual)
clone_pinned()
RUN_ID=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT=REPO/'results'/f'{PLATFORM}-h0-{COMMIT[:7]}-{RUN_ID}'
OUT.mkdir(parents=True,exist_ok=False)
(OUT/'gpu-inventory.txt').write_text(gpu)
(OUT/'run-context.json').write_text(json.dumps({'commit':COMMIT,'platform':PLATFORM,'run_id':RUN_ID},indent=2))

def logged(argv, filename, *, env=None, timeout=3600):
    path=OUT/filename
    start=time.monotonic()
    with path.open('w') as log:
        proc=subprocess.Popen(list(map(str,argv)),cwd=REPO,env=env,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
        try:
            while proc.poll() is None:
                elapsed=time.monotonic()-start
                if elapsed > timeout:
                    raise TimeoutError(f'Wall-time cap reached: {filename}')
                print(f'{filename}: running, {int(elapsed)} seconds elapsed',flush=True)
                time.sleep(20)
        except BaseException:
            try:
                os.killpg(proc.pid,signal.SIGTERM)
                proc.wait(timeout=20)
            except (ProcessLookupError,subprocess.TimeoutExpired):
                try: os.killpg(proc.pid,signal.SIGKILL)
                except ProcessLookupError: pass
            raise
    print(filename,'exit:',proc.returncode)
    if proc.returncode:
        print(path.read_text(errors='replace')[-3000:])
    return proc.returncode


## 2. Install and record the isolated environment


In [ ]:
# Reuse the notebook's CUDA-enabled torch; keep changes in a separate virtualenv.
VENV=Path('/tmp/inferstack-native-env')
subprocess.run([sys.executable,'-m','venv','--system-site-packages',str(VENV)],check=True)
PY=VENV/'bin/python'
ENV={**os.environ,'PATH':str(VENV/'bin')+os.pathsep+os.environ['PATH'],
     'PYTHONPATH':str(REPO/'src'),'INFERSTACK_PROVIDER':'colab','CUDA_VISIBLE_DEVICES':'0'}
assert logged([PY,'-m','pip','install','--require-hashes','-r','requirements-dev.lock'],
              'install-core.log',env=ENV,timeout=2700)==0
assert logged([PY,'-m','pip','install','--no-cache-dir','-e','.[dev,native,hf,postgres]',
               'transformers[kernels]==5.22.2'], 'install-native.log',env=ENV,timeout=2700)==0
# This package enables isolated-build tests only if the notebook host allows namespaces.
if os.geteuid()==0:
    logged(['apt-get','update','-qq'],'apt-update.log',timeout=600)
    logged(['apt-get','install','-y','bubblewrap'],'apt-bubblewrap.log',timeout=600)
preflight="""import json, torch, triton, transformers
from inferstack.execution.sandbox import Sandbox
assert torch.cuda.is_available(), 'CUDA torch unavailable in the run environment'
assert torch.cuda.is_bf16_supported(), 'This native profile requires BF16'
print(json.dumps({'torch':torch.__version__, 'cuda':torch.version.cuda,
                  'transformers':transformers.__version__, 'triton':triton.__version__,
                  'gpu':torch.cuda.get_device_name(0), 'sandbox':Sandbox().describe()},indent=2))
"""
assert logged([PY,'-c',preflight],'preflight.log',env=ENV,timeout=180)==0
print((OUT/'preflight.log').read_text())
(OUT/'pip-freeze.txt').write_text(subprocess.check_output([str(PY),'-m','pip','freeze'],env=ENV,text=True))
assert logged([PY,'-c',"import importlib.util; assert all(importlib.util.find_spec(m) is None for m in ('vllm','sglang')), 'Use a fresh native-only runtime without vLLM/SGLang'"], 'engine-environment.log',env=ENV,timeout=60)==0


## 3. Run qualification


In [ ]:
# Fixed public model from the Foundry manifest; no optimizer-provider key is needed.
manifest=json.loads((REPO/'examples/model_foundry/factorylm-2b/candidate/manifest.json').read_text())
assert manifest['source']['uri']=='Qwen/Qwen3.5-2B-Base'
assert manifest['source']['revision']=='b1485b2fa6dfa1287294f269f5fb618e03d52d7c'
(OUT/'model-manifest.json').write_text(json.dumps(manifest,indent=2))
# Maximum three hours for the consolidated run. This is a test, not a promised pass.
run_exit=logged([PY,'-u','scripts/v2_qualify.py','--out',str(OUT.relative_to(REPO))],
                'qualification-runner.log',env=ENV,timeout=3*60*60)
print('Qualification process exit:',run_exit,'(0=qualified, 2=not_qualified; inspect the report)')


## 4. Read the verdict


In [ ]:
report_path=OUT/'qualification.json'
if report_path.exists():
    q=json.loads(report_path.read_text())
    print('Verdict:',q['verdict'])
    print('GPU profile:',q.get('gpu_profile'))
    print('Blocking gates:',q['blocking'])
    for name,result in q['gates'].items():
        print(name, result['status'])
else:
    print('No final report: inspect qualification-runner.log and the last completed gate log.')
print('A native smoke pass is liveness evidence. Performance claims need repeated comparisons; this run is not cloud/FactoryOS qualification.')


## 5. Download evidence


In [ ]:
import shutil
archive_base=BASE/f'inferstack-{PLATFORM}-evidence-{RUN_ID}'
zip_path=shutil.make_archive(str(archive_base),'zip',root_dir=OUT.parent,base_dir=OUT.name)
print('Evidence archive:',zip_path)
if PLATFORM=='colab':
    from google.colab import files
    files.download(zip_path)
else:
    from IPython.display import FileLink, display
    display(FileLink(zip_path))
    print('You can also download the ZIP from the notebook file browser under /kaggle/working.')
